# Screened C0: 0.5B, parent seed 17

Continuation seed **42** is retained from the B recipe. This notebook launches a matched B/C0 pilot and development evaluation. Full training has not been run when this notebook is created.

Upload the ZIP into `/scratch/sshind38/capstone_c0_c1_work/`, then run the next cell. It verifies/freezes the successful preview and submits one A100 job. See README for scope and interruption recovery.


In [ ]:
from pathlib import Path
import subprocess
import zipfile

WORK = Path('/scratch/sshind38/capstone_c0_c1_work')
NAME = 'C0_Screened_05B_Seed17_20261110'
TRAIN_PYTHON = Path('/scratch/sshind38/capstone_approach_b/venv_b_training_py312/bin/python')
ATTEMPT = 'pilot01'
archive = WORK / f'{NAME}.zip'
if not archive.is_file():
    raise FileNotFoundError(f'Upload {archive.name} into {WORK} first')
if not TRAIN_PYTHON.is_file():
    raise FileNotFoundError(f'Existing training Python is missing: {TRAIN_PYTHON}')
with zipfile.ZipFile(archive) as package:
    bad_member = package.testzip()
    if bad_member:
        raise ValueError(f'Archive is damaged: {bad_member}')
    for member in package.infolist():
        target = (WORK / member.filename).resolve()
        if (not target.is_relative_to(WORK.resolve())
                or Path(member.filename).parts[0] != NAME):
            raise ValueError(f'Unexpected archive path: {member.filename}')
        if member.is_dir():
            continue
        content = package.read(member)
        if target.exists():
            if target.read_bytes() != content:
                raise FileExistsError(f'Different existing file preserved: {target}')
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as stream:
                stream.write(content)
runner = WORK / NAME / 'screened_c0.py'
for action in ['prepare', 'submit']:
    subprocess.run([str(TRAIN_PYTHON), '-B', str(runner), action,
                    '--work', str(WORK), '--attempt', ATTEMPT],
                   cwd=WORK, check=True)


## Check progress

Run while the Slurm job is pending/running. A submitted job ID is not a completion receipt.


In [ ]:
from pathlib import Path
import subprocess

WORK = Path('/scratch/sshind38/capstone_c0_c1_work')
NAME = 'C0_Screened_05B_Seed17_20261110'
TRAIN_PYTHON = '/scratch/sshind38/capstone_approach_b/venv_b_training_py312/bin/python'
subprocess.run([TRAIN_PYTHON, '-B', str(WORK / NAME / 'screened_c0.py'),
                'status', '--work', str(WORK), '--attempt', 'pilot01'],
               cwd=WORK, check=True)


## View job state and recent logs


In [ ]:
from pathlib import Path
import json
import subprocess

WORK = Path('/scratch/sshind38/capstone_c0_c1_work')
location = json.loads((WORK / 'C0_Screened_05B_Seed17_20261110' / 'RUN_LOCATION.json').read_text())
submissions = sorted((Path(location['output_directory']) / 'submissions' / 'pilot01').glob('*.json'))
if not submissions:
    raise FileNotFoundError('No submitted job receipt exists yet')
job = json.loads(submissions[-1].read_text())
print('Job:', job['job_id'])
subprocess.run(['squeue', '-j', job['job_id']], check=False)
for label in ['stdout', 'stderr']:
    path = Path(job[label])
    print(f'\n{label}: {path}')
    if path.is_file():
        print('\n'.join(path.read_text(errors='replace').splitlines()[-35:]))


## Finish

Wait for `screened_c0_pilot_complete: true`. The status output gives `completion_report.json` and the results ZIP path. Upload that results ZIP to this conversation for review.

Results apply to the adopted conservative screen, with inherited parent/development exposure disclosed. Reserved final tests and C1 remain separate.
